# 00 · Environment setup

Run this notebook **once per project**, not once per session. It

1. mounts Google Drive and checks the project folders,
2. records the Colab environment (Python, PyTorch, RAM, GPU, disk, package versions),
3. checks which GPU kernels work (this explains one design decision, see the cell below),
4. copies the MVTec AD subset (7 categories, about 1.9 GB) from Drive to Colab's local disk and verifies it,
5. builds `mvtec_7cat.tar` on Drive. Every other notebook restores the dataset from this single file, which is much faster than copying thousands of small files from Drive.

Code lives in the GitHub repo and is cloned by the setup cell of the other notebooks. Results are written to Drive.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_PROJECT = Path("/content/drive/MyDrive/industrial-anomaly-detection")
if not DRIVE_PROJECT.exists():
    raise FileNotFoundError(f"Project folder not found: {DRIVE_PROJECT}")

print("Project found:", DRIVE_PROJECT)
print("\nContents:")
for item in sorted(DRIVE_PROJECT.iterdir()):
    print("[dir] " if item.is_dir() else "[file]", item.name)

In [ ]:
# Persistent storage on Google Drive
DRIVE_DATASET = DRIVE_PROJECT / "datasets" / "mvtec_anomaly_detection"

# Temporary Colab workspace (wiped whenever the runtime is recycled)
WORKSPACE = Path("/content/industrial-anomaly-detection")
LOCAL_DATASET = WORKSPACE / "datasets" / "mvtec_anomaly_detection"

CATEGORIES = ["bottle", "capsule", "carpet", "metal_nut", "screw", "toothbrush", "zipper"]

print("Drive dataset:  ", DRIVE_DATASET)
print("Local dataset:  ", LOCAL_DATASET)

In [ ]:
# Check the Drive folders and the seven category folders; create the report folders if missing.
for folder in ["datasets", "notebooks", "experiments", "models"]:
    print(f"{'OK' if (DRIVE_PROJECT / folder).is_dir() else 'MISSING'}: {folder}")

print("\nDataset exists:", DRIVE_DATASET.is_dir())
for category in CATEGORIES:
    print(f"{'OK' if (DRIVE_DATASET / category).is_dir() else 'MISSING'}: {category}")

for folder in ["reports/dataset_audit", "reports/metrics", "reports/figures", "reports/visualizations"]:
    (DRIVE_PROJECT / folder).mkdir(parents=True, exist_ok=True)
print("\nReport folders ready.")

## Environment record

In [ ]:
import platform
import psutil
import torch

print("===== SYSTEM =====")
print("Python: ", platform.python_version())
print("PyTorch:", torch.__version__)

print("\n===== RAM =====")
ram = psutil.virtual_memory()
print(f"Total: {ram.total / 1024**3:.2f} GB | available: {ram.available / 1024**3:.2f} GB")

print("\n===== GPU =====")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0), f"| memory: {gpu.total_memory / 1024**3:.2f} GB")
else:
    print("No GPU (CPU runtime)")

print("\n===== DISK =====")
disk = psutil.disk_usage("/content")
print(f"Total: {disk.total / 1024**3:.2f} GB | free: {disk.free / 1024**3:.2f} GB")

In [ ]:
import importlib

for package in ["torch", "torchvision", "numpy", "pandas", "matplotlib", "PIL", "cv2", "sklearn", "yaml", "tqdm", "psutil"]:
    try:
        module = importlib.import_module(package)
        print(f"OK: {package} - {getattr(module, '__version__', 'installed')}")
    except ImportError:
        print(f"MISSING: {package}")

## GPU kernel check

On our Colab image (Python 3.13, PyTorch 2.14, T4), `torch.linalg.inv` on a **CUDA** tensor failed with
`no kernel image is available for execution on the device`. That is why `PaDiM.fit` computes the covariance
inverse on the CPU. This cell reproduces the check. It needs a GPU runtime and is only informative: the
result depends on the Colab image of the day.

In [ ]:
if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("Compute capability:", torch.cuda.get_device_capability(0))
    print("Supported CUDA architectures:", torch.cuda.get_arch_list())
    for dtype in (torch.float32, torch.float64):
        try:
            torch.linalg.inv(torch.randn(8, 8, device="cuda", dtype=dtype))
            torch.cuda.synchronize()
            print(f"{dtype}: GPU inversion PASSED")
        except Exception as error:
            print(f"{dtype}: GPU inversion FAILED -> {type(error).__name__}: {error}")
else:
    print("Skipped: no GPU on this runtime.")

## Copy the dataset to Colab's local disk

Reading thousands of small images from Drive is slow, so experiments read from `/content`. `rsync` is safe to
re-run: it only copies what is missing.

In [ ]:
import shutil
import subprocess

if not DRIVE_DATASET.exists():
    raise FileNotFoundError(f"Dataset not found: {DRIVE_DATASET}")

LOCAL_DATASET.mkdir(parents=True, exist_ok=True)
if shutil.which("rsync"):
    subprocess.run(["rsync", "-ah", "--info=progress2", f"{DRIVE_DATASET}/", f"{LOCAL_DATASET}/"], check=True)
else:
    print("rsync unavailable, using a Python copy.")
    shutil.copytree(DRIVE_DATASET, LOCAL_DATASET, dirs_exist_ok=True)
print("\nDataset transfer completed.")

In [ ]:
# Verify the copy: same number of files and same total size in bytes.
def dataset_summary(directory: Path) -> dict:
    files = [p for p in directory.rglob("*") if p.is_file()]
    return {"file_count": len(files), "total_bytes": sum(p.stat().st_size for p in files)}

drive_summary = dataset_summary(DRIVE_DATASET)
local_summary = dataset_summary(LOCAL_DATASET)
print("Google Drive:", drive_summary)
print("Colab:       ", local_summary)

assert drive_summary["file_count"] == local_summary["file_count"], "File counts do not match!"
assert drive_summary["total_bytes"] == local_summary["total_bytes"], "Total sizes do not match!"
print("\nDataset transfer verification PASSED.")

## Build the dataset archive (once)

PNG files are already compressed, so a plain `.tar` is enough. It is written under a temporary name and renamed
only when finished, so an interrupted upload can never leave a half-written file that looks valid.
`ensure_dataset()` in the other notebooks only trusts the archive if all seven categories extract correctly,
and falls back to the slow file-by-file copy otherwise.

In [ ]:
ARCHIVE = DRIVE_PROJECT / "datasets" / "mvtec_7cat.tar"
tmp = ARCHIVE.with_name(ARCHIVE.name + ".tmp")

if ARCHIVE.exists():
    print("Archive already exists.")
else:
    subprocess.run(["tar", "-cf", str(tmp), "-C", str(LOCAL_DATASET.parent), "mvtec_anomaly_detection"], check=True)
    shutil.move(str(tmp), str(ARCHIVE))
print(f"{ARCHIVE.name}: {ARCHIVE.stat().st_size / 1e9:.2f} GB")